# 📊 Data Pipeline - E-commerce de Tecnología

Este proyecto implementa un flujo completo de generación y carga de datos en **Google BigQuery**, separando responsabilidades entre **Data Engineer 1 (datos maestros)** y **Data Engineer 2 (datos transaccionales)**.



## 👤 Data Engineer 1 — Datos Maestros

Responsable de generar las tablas base del sistema, que servirán como fuente para los datos transaccionales.

### Tablas generadas

- **customers** → Información de clientes  
- **categories** → Catálogo de categorías tecnológicas  
- **products** → Productos disponibles en el e-commerce  

### Características

- Mínimo 500 clientes  
- Mínimo 70 productos  
- Coste menor que precio en la mayoría  
- Stock siempre ≥ 0  
- Datos realistas con Faker  
- IDs únicos  

---

## 🔄 Data Engineer 2 — Datos Transaccionales

Construye la capa de negocio a partir de los datos maestros.

### Tablas generadas

- **orders** → Pedidos realizados por clientes  
- **order_items** → Productos dentro de cada pedido  
- **payments** → Pagos asociados a pedidos  
- **reviews** → Reseñas de productos entregados  

### Reglas de negocio

- Cada pedido usa un `customer_id` válido  
- Cada línea usa un `product_id` válido  
- El importe del pago coincide con el total del pedido  
- Las reseñas solo existen para pedidos entregados  
- Se preserva el precio histórico (`unit_price`)  

---


In [10]:
import os
import pandas as pd
import numpy as np
from faker import Faker
from datetime import timedelta
from google.cloud import bigquery
from google.oauth2 import service_account

fake = Faker()
np.random.seed(42)


# ============================================================
# 0. SETUP BIGQUERY SIN .ENV
# ============================================================

# ID del proyecto en Google Cloud
PROJECT_ID = "whiteberry-network"

# Nombre del dataset en BigQuery
DATASET_ID = "whiteberry"

# Ruta directa al JSON de credenciales
CREDENTIALS_PATH = r"C:\Users\anaco\OneDrive\Documentos\GitHub\whiteberry\credentials\service-account.json"

# Validamos que el archivo existe
if not os.path.exists(CREDENTIALS_PATH):
    raise FileNotFoundError(f"No se encontró el archivo de credenciales: {CREDENTIALS_PATH}")

# Creamos credenciales desde el JSON
credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH
)

# Creamos cliente de BigQuery
client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

# Creamos dataset si no existe
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "EU"

client.create_dataset(dataset_ref, exists_ok=True)

print(f"Dataset {DATASET_ID} creado o ya existente")


# ============================================================
# ENGINEER A - DATOS MAESTROS
# customers, categories, products
# ============================================================

# -------------------------------
# 1. CATEGORIES
# -------------------------------

categories = [
    {"category_id": 1, "name": "Laptops", "description": "Portable computers"},
    {"category_id": 2, "name": "Smartphones", "description": "Mobile phones"},
    {"category_id": 3, "name": "Tablets", "description": "Tablet devices"},
    {"category_id": 4, "name": "Accessories", "description": "Chargers, cables and peripherals"},
    {"category_id": 5, "name": "Gaming", "description": "Gaming consoles and gear"},
    {"category_id": 6, "name": "Audio", "description": "Headphones and speakers"},
    {"category_id": 7, "name": "Monitors", "description": "Screens and displays"},
    {"category_id": 8, "name": "Components", "description": "Computer hardware components"},
]

categories_df = pd.DataFrame(categories)


# -------------------------------
# 2. CUSTOMERS
# -------------------------------

N_CUSTOMERS = 600

countries_cities = {
    "Spain": ["Madrid", "Barcelona", "Valencia", "Seville"],
    "France": ["Paris", "Lyon", "Marseille"],
    "Germany": ["Berlin", "Munich", "Hamburg"],
    "Italy": ["Rome", "Milan", "Turin"],
    "United Kingdom": ["London", "Manchester", "Birmingham"],
}

channels = ["organic", "paid_ads", "social_media", "referral"]

customers = []

for customer_id in range(1, N_CUSTOMERS + 1):
    country = np.random.choice(list(countries_cities.keys()))
    city = np.random.choice(countries_cities[country])

    customers.append({
        "customer_id": customer_id,
        "firstname": fake.first_name(),
        "lastname": fake.last_name(),
        "email": fake.unique.email(),
        "country": country,
        "city": city,
        "registration_date": fake.date_between(start_date="-3y", end_date="today"),
        "source_channel": np.random.choice(channels)
    })

customers_df = pd.DataFrame(customers)


# -------------------------------
# 3. PRODUCTS
# -------------------------------

N_PRODUCTS = 100

brands = ["Apple", "Samsung", "Dell", "HP", "Lenovo", "Asus", "Sony", "LG", "Logitech", "MSI"]

category_products = {
    1: ["Laptop", "Ultrabook", "Notebook"],
    2: ["Smartphone", "Mobile Phone"],
    3: ["Tablet", "iPad"],
    4: ["Keyboard", "Mouse", "Charger", "USB-C Cable", "Webcam"],
    5: ["Gaming Console", "Gaming Mouse", "Gaming Keyboard", "Controller"],
    6: ["Headphones", "Bluetooth Speaker", "Earbuds"],
    7: ["Monitor", "Curved Monitor", "4K Display"],
    8: ["SSD", "RAM", "GPU", "Motherboard", "Processor"],
}

products = []

for product_id in range(1, N_PRODUCTS + 1):
    category_id = np.random.choice(categories_df["category_id"])
    brand = np.random.choice(brands)
    product_type = np.random.choice(category_products[category_id])

    price = round(np.random.uniform(50, 2000), 2)

    # Coste menor que precio en la mayoría de productos
    if np.random.rand() < 0.85:
        cost = round(price * np.random.uniform(0.4, 0.8), 2)
    else:
        cost = round(price * np.random.uniform(0.85, 1.05), 2)

    products.append({
        "product_id": product_id,
        "category_id": int(category_id),
        "name": f"{brand} {product_type}",
        "created_at": fake.date_between(start_date="-2y", end_date="today"),
        "price": price,
        "cost": cost,
        "stock": int(np.random.randint(0, 250)),
        "is_available": bool(np.random.choice([True, False], p=[0.85, 0.15]))
    })

products_df = pd.DataFrame(products)


# -------------------------------
# VALIDACIONES ENGINEER A
# -------------------------------

assert customers_df["customer_id"].is_unique
assert products_df["product_id"].is_unique
assert categories_df["category_id"].is_unique
assert len(customers_df) >= 500
assert len(products_df) >= 70
assert (products_df["stock"] >= 0).all()
assert (products_df["price"] > products_df["cost"]).mean() > 0.7
assert products_df["category_id"].isin(categories_df["category_id"]).all()


# ============================================================
# ENGINEER B - DATOS TRANSACCIONALES
# orders, order_items, payments, reviews
# ============================================================

valid_products = products_df[
    (products_df["is_available"] == True) &
    (products_df["stock"] > 0)
].copy()

assert len(valid_products) > 0


# -------------------------------
# 4. ORDERS
# -------------------------------

def generate_orders(customers_df, n_orders=2000):
    statuses = ["pending", "confirmed", "shipped", "delivered", "cancelled", "returned"]
    status_probs = [0.10, 0.15, 0.20, 0.40, 0.10, 0.05]

    orders = []

    for order_id in range(1, n_orders + 1):
        customer_id = np.random.choice(customers_df["customer_id"])
        status = np.random.choice(statuses, p=status_probs)

        order_date = fake.date_between(start_date="-2y", end_date="today")

        shipped_date = pd.NaT
        delivered_date = pd.NaT

        if status in ["shipped", "delivered", "returned"]:
            shipped_date = order_date + timedelta(days=np.random.randint(1, 5))

        if status in ["delivered", "returned"]:
            delivered_date = shipped_date + timedelta(days=np.random.randint(1, 8))

        orders.append({
            "order_id": order_id,
            "customer_id": int(customer_id),
            "order_date": order_date,
            "shipped_date": shipped_date,
            "delivered_date": delivered_date,
            "status": status
        })

    return pd.DataFrame(orders)


orders_df = generate_orders(customers_df, n_orders=2000)


# -------------------------------
# 5. ORDER_ITEMS
# -------------------------------

def generate_order_items(orders_df, valid_products, target_lines=4500):
    order_items = []
    order_item_id = 1

    # Primero aseguramos mínimo 1 línea por pedido
    for _, order in orders_df.iterrows():
        product = valid_products.sample(n=1).iloc[0]

        quantity = int(np.random.randint(1, 5))
        unit_price = float(product["price"])

        discount_rate = np.random.choice([0, 0.05, 0.10], p=[0.75, 0.15, 0.10])
        discount_amount = round(quantity * unit_price * discount_rate, 2)

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": int(order["order_id"]),
            "product_id": int(product["product_id"]),
            "quantity": quantity,
            "unit_price": round(unit_price, 2),
            "discount_amount": discount_amount
        })

        order_item_id += 1

    # Luego añadimos líneas extra hasta llegar a unas 4500 líneas
    extra_lines = target_lines - len(order_items)

    for _ in range(extra_lines):
        order = orders_df.sample(n=1).iloc[0]
        product = valid_products.sample(n=1).iloc[0]

        quantity = int(np.random.randint(1, 5))
        unit_price = float(product["price"])

        discount_rate = np.random.choice([0, 0.05, 0.10], p=[0.75, 0.15, 0.10])
        discount_amount = round(quantity * unit_price * discount_rate, 2)

        order_items.append({
            "order_item_id": order_item_id,
            "order_id": int(order["order_id"]),
            "product_id": int(product["product_id"]),
            "quantity": quantity,
            "unit_price": round(unit_price, 2),
            "discount_amount": discount_amount
        })

        order_item_id += 1

    return pd.DataFrame(order_items)


order_items_df = generate_order_items(
    orders_df,
    valid_products,
    target_lines=4500
)


# -------------------------------
# 6. PAYMENTS
# -------------------------------

def generate_payments(orders_df, order_items_df):
    payments = []
    payment_id = 1

    payment_methods = ["credit_card", "paypal", "bank_transfer"]

    order_totals = (
        order_items_df
        .assign(total=lambda x: x["quantity"] * x["unit_price"] - x["discount_amount"])
        .groupby("order_id")["total"]
        .sum()
        .reset_index()
    )

    for _, order in orders_df.iterrows():
        order_id = int(order["order_id"])
        status = order["status"]

        total = order_totals.loc[
            order_totals["order_id"] == order_id,
            "total"
        ].iloc[0]

        if status in ["confirmed", "shipped", "delivered"]:
            payment_status = "completed"
            payment_type = "payment"

        elif status == "pending":
            payment_status = np.random.choice(["pending", "failed"], p=[0.7, 0.3])
            payment_type = "payment"

        elif status == "cancelled":
            payment_status = np.random.choice(["failed", "refunded"], p=[0.6, 0.4])
            payment_type = "refund" if payment_status == "refunded" else "payment"

        elif status == "returned":
            payment_status = "refunded"
            payment_type = "refund"

        payments.append({
            "payment_id": payment_id,
            "order_id": order_id,
            "payment_date": order["order_date"] + timedelta(days=np.random.randint(0, 3)),
            "payment_method": np.random.choice(payment_methods),
            "payment_type": payment_type,
            "amount": round(total, 2),
            "status": payment_status
        })

        payment_id += 1

    return pd.DataFrame(payments)


payments_df = generate_payments(orders_df, order_items_df)


# -------------------------------
# 7. REVIEWS
# -------------------------------

def generate_reviews(orders_df, order_items_df, review_rate=0.35):
    reviews = []

    delivered_orders = orders_df[
        orders_df["status"] == "delivered"
    ][["order_id", "delivered_date"]]

    eligible_items = order_items_df.merge(delivered_orders, on="order_id")

    review_sample = eligible_items.sample(
        frac=review_rate,
        random_state=42
    )

    for review_id, (_, row) in enumerate(review_sample.iterrows(), start=1):
        reviews.append({
            "review_id": review_id,
            "order_item_id": int(row["order_item_id"]),
            "review_date": row["delivered_date"] + timedelta(days=np.random.randint(1, 30)),
            "rating": int(np.random.choice([1, 2, 3, 4, 5], p=[0.05, 0.10, 0.20, 0.35, 0.30])),
            "comment": fake.sentence()
        })

    return pd.DataFrame(reviews)


reviews_df = generate_reviews(
    orders_df,
    order_items_df,
    review_rate=0.35
)


# -------------------------------
# VALIDACIONES ENGINEER B
# -------------------------------

assert orders_df["customer_id"].isin(customers_df["customer_id"]).all()
assert order_items_df["order_id"].isin(orders_df["order_id"]).all()
assert order_items_df["product_id"].isin(products_df["product_id"]).all()
assert payments_df["order_id"].isin(orders_df["order_id"]).all()
assert reviews_df["order_item_id"].isin(order_items_df["order_item_id"]).all()

assert orders_df["order_id"].isin(order_items_df["order_id"]).all()

assert (
    orders_df.dropna(subset=["shipped_date"])
    .eval("order_date <= shipped_date")
).all()

assert (
    orders_df.dropna(subset=["delivered_date"])
    .eval("shipped_date <= delivered_date")
).all()

assert orders_df.loc[
    orders_df["status"] == "pending",
    "delivered_date"
].isna().all()

assert orders_df.loc[
    orders_df["status"] == "cancelled",
    "delivered_date"
].isna().all()

order_totals = (
    order_items_df
    .assign(total=lambda x: x["quantity"] * x["unit_price"] - x["discount_amount"])
    .groupby("order_id")["total"]
    .sum()
    .reset_index()
)

payment_check = payments_df.merge(order_totals, on="order_id")
payment_check["diff"] = abs(payment_check["amount"] - payment_check["total"])

assert (payment_check["diff"] < 0.01).all()

review_check = (
    reviews_df
    .merge(order_items_df, on="order_item_id")
    .merge(orders_df, on="order_id")
)

assert (review_check["status"] == "delivered").all()


# ============================================================
# BIGQUERY - CREAR TABLAS
# ============================================================

customers_schema = [
    bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("firstname", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("lastname", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("email", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("country", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("city", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("registration_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("source_channel", "STRING", mode="REQUIRED"),
]

categories_schema = [
    bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
]

products_schema = [
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("created_at", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("price", "FLOAT64", mode="REQUIRED"),
    bigquery.SchemaField("cost", "FLOAT64", mode="REQUIRED"),
    bigquery.SchemaField("stock", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("is_available", "BOOL", mode="REQUIRED"),
]

orders_schema = [
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("shipped_date", "DATE", mode="NULLABLE"),
    bigquery.SchemaField("delivered_date", "DATE", mode="NULLABLE"),
    bigquery.SchemaField("status", "STRING", mode="REQUIRED"),
]

order_items_schema = [
    bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("quantity", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("unit_price", "FLOAT64", mode="REQUIRED"),
    bigquery.SchemaField("discount_amount", "FLOAT64", mode="REQUIRED"),
]

payments_schema = [
    bigquery.SchemaField("payment_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("payment_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("payment_method", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("payment_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("amount", "FLOAT64", mode="REQUIRED"),
    bigquery.SchemaField("status", "STRING", mode="REQUIRED"),
]

reviews_schema = [
    bigquery.SchemaField("review_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("review_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("rating", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("comment", "STRING", mode="NULLABLE"),
]

tables = {
    "customers": customers_schema,
    "categories": categories_schema,
    "products": products_schema,
    "orders": orders_schema,
    "order_items": order_items_schema,
    "payments": payments_schema,
    "reviews": reviews_schema,
}

for table_name, schema in tables.items():
    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    table = bigquery.Table(table_ref, schema=schema)
    client.create_table(table, exists_ok=True)
    print(f"Tabla creada o ya existente: {table_name}")


# ============================================================
# BIGQUERY - CARGAR DATOS
# ============================================================

# Convertir fechas a DATE compatible con BigQuery
customers_df["registration_date"] = pd.to_datetime(customers_df["registration_date"]).dt.date
products_df["created_at"] = pd.to_datetime(products_df["created_at"]).dt.date

orders_df["order_date"] = pd.to_datetime(orders_df["order_date"]).dt.date
orders_df["shipped_date"] = pd.to_datetime(orders_df["shipped_date"], errors="coerce").dt.date
orders_df["delivered_date"] = pd.to_datetime(orders_df["delivered_date"], errors="coerce").dt.date

payments_df["payment_date"] = pd.to_datetime(payments_df["payment_date"]).dt.date
reviews_df["review_date"] = pd.to_datetime(reviews_df["review_date"]).dt.date

dataframes = {
    "customers": customers_df,
    "categories": categories_df,
    "products": products_df,
    "orders": orders_df,
    "order_items": order_items_df,
    "payments": payments_df,
    "reviews": reviews_df,
}

for table_name, df in dataframes.items():
    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_TRUNCATE",
        schema=tables[table_name]
    )

    job = client.load_table_from_dataframe(
        df,
        table_ref,
        job_config=job_config
    )

    job.result()

    print(f"{len(df)} filas cargadas en {table_name}")


# ============================================================
# RESUMEN FINAL
# ============================================================

print("Proceso completado correctamente")
print("Engineer A:")
print("customers:", len(customers_df))
print("categories:", len(categories_df))
print("products:", len(products_df))

print("Engineer B:")
print("orders:", len(orders_df))
print("order_items:", len(order_items_df))
print("payments:", len(payments_df))
print("reviews:", len(reviews_df))

Dataset whiteberry creado o ya existente
Tabla creada o ya existente: customers
Tabla creada o ya existente: categories
Tabla creada o ya existente: products
Tabla creada o ya existente: orders
Tabla creada o ya existente: order_items
Tabla creada o ya existente: payments
Tabla creada o ya existente: reviews


c:\Users\anaco\OneDrive\Documentos\GitHub\whiteberry\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


600 filas cargadas en customers


c:\Users\anaco\OneDrive\Documentos\GitHub\whiteberry\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


8 filas cargadas en categories


...

4500 filas cargadas en order_items


c:\Users\anaco\OneDrive\Documentos\GitHub\whiteberry\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


2000 filas cargadas en payments


c:\Users\anaco\OneDrive\Documentos\GitHub\whiteberry\venv\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


604 filas cargadas en reviews
Proceso completado correctamente
Engineer A:
customers: 600
categories: 8
products: 100
Engineer B:
orders: 2000
order_items: 4500
payments: 2000
reviews: 604
